In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import random
import string

from collections import Counter


In [3]:
RED = "\033[31;1m"
GREEN = "\033[32;1m"
YELLOW = "\033[33;1m"
BLUE = "\033[34;1m"
PURPLE = "\033[35;1m"
CYAN = "\033[36;1m"
WHITE = "\033[37;1m"

In [4]:
#Random seeds for reproducibility
torch.manual_seed(50)
np.random.seed(50)
random.seed(50)

In [5]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2) -> None:
        super(RNN, self).__init__()
        
        self.hidden_layer = hidden_size
        self.num_layers = num_layers
        
        #Embedding Layer to convert to characters to dense vectors
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        
        #LSTM layer (more stable than vanilla RNN)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        
        #Output layer to predict next character
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=True):
        # x shape: (batch_size, sequence_length)
        embeddings = self.embeddings(x) # (batch_size, sequence_length, embed_size)
        
        #Pass through LSTM
        output, hidden = self.lstm(embeddings, hidden)
        
        #Apply linear layer to get character probability
        output = self.fc(output)
        
        return output, hidden
    
    def init_hidden(self, batch_size, device):
        """Initialize the hidden state"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        
        return (h0, c0)

In [6]:
class TextProcessor:
    def __init__(self) -> None:
        self.char_to_idx = {}
        self.idx_to_char = {}
        self.vocab_size = 0
        
    def build_vocab(self, text):
        """Building character vocabulary from text"""
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        
        
        #Create Mapping
        self.char_to_idx = {char:idx for idx, char in enumerate(chars)}
        self.idx_to_char = {idx:char for idx, char in enumerate(chars)}
        
        print(f"{BLUE}Vocabulary Size: {self.vocab_size}")
        print(f"{WHITE}Characters: {' '.join(chars[:50])}")
        
    def text_to_indices(self, text):
        return [self.char_to_idx[char] for char in text if char in self.char_to_idx]
    
    def indices_to_text(self, text):
        return [self.idx_to_char[idx] for idx in text]

In [7]:
def load_and_preprocess_text(file_path):
    try:
        with open(file_path, mode="r", encoding="utf-8") as file:
            text = file.read()
    except FileNotFoundError:
        print(f"{RED}File not found. Load the file again")
        raise FileNotFoundError("File does not exist")
    
    text = text.strip()
    
    print(f"{PURPLE}Text Length: {len(text)} characters")
    print(f"Sample Text: {text[:100]}...")
    
    return text

In [8]:
def create_sequence(text_indices, seq_length):
    sequences = []
    targets = []
    
    for i in range(len(text_indices) - seq_length):
        #Input Sequence
        seq = text_indices[i:i+seq_length]
        #Target is next character
        target = text_indices[i+1: i+seq_length+1]
        
        sequences.append(seq)
        targets.append(target)
        
    return np.array(sequences), np.array(targets)

In [9]:
def train_model(model, data_loader, criterion, optimizer, device, epochs=50):
    """
    Train the RNN Model
    """
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        #Let the model hiddden state initialization
        hidden = None
        
        for batch_idx, (sequences, targets) in enumerate(data_loader):
            sequences = sequences.to(device)
            targets = targets.to(device)
            
            #Zero gradients
            optimizer.zero_grad()
            
            #Forward Propogation
            output, hidden = model(sequences, hidden)
            
            #Detach hidden state to prevent backprop through entire sequence
            if hidden is not None:
                hidden = (hidden[0].detach(), hidden[1].detach())
                
            #Calculating Loss
            #Reshape for cross-entropy: (batch_size * seq_length, vocab_size)
            output = output.reshape(-1, output.size(-1))
            targets = targets.reshape(-1)
            loss = criterion(output, targets)
            
            #Backward Propogation
            loss.backward()
            
            #Clipping gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            
            #Update weights
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(data_loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"{CYAN}Epoch [{epoch + 1}/{epoch}], Loss: {avg_loss:.4f}")

    return losses
    

In [10]:
def generate_model(model, processor, seed_text, length=200, temperature=0.8, device="gpu"):
    model.eval()
    
    #Converting seed text to indices
    current_seq = processor.text_to_indices(seed_text)
    generated = current_seq.copy()
    
    #Initialize the hidden state
    hidden = model.init_hidden(1, device)
    
    with torch.no_grad():
        for _ in range(0, length):
            seq_length = min(len(current_seq), 50)
            input_seq = torch.tensor([current_seq[-seq_length:]], dtype=torch.long).to(device)
            
            #Getting Predictions
            output, hidden = model(input_seq, hidden)
            
            #Geting the last output (prediction for next character)
            last_output = output[0 -1, :] #vocab size
            
            #Applying the temperature for diverse generation
            last_output = last_output / temperature
            
            #Converting it into probability
            probabilities = torch.softmax(last_output, dim=0)
            
            #Sample for next distribution
            next_char_idx = torch.multinomial(probabilities, 1).item()
            
            #Adding to sequence
            current_seq.append(next_char_idx)
            generated.append(next_char_idx)
    
    #Convert back to text
    return processor.indices_to_text(generated)

In [ ]:
def main():
    #Hyperparameters
    SEQUENCE_LENGTH = 50 #Length of Input Sequence
    EMBED_SIZE = 128 #Embedding dimension
    HIDDEN_SIZE = 256 #LSTM Hidden Size
    NUM_LAYERS = 2 #Number of LSTM Layers
    BATCH_SIZE = 64 #Batch Size for Training
    LEARNING_RATE = 0.002 #Learning Rate
    EPOCHS = 100 #Number of training epochs
    
    #Setting Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"{BLUE}Using Device: {device}")
    
    #Loading and preprocessing the data
    text = load_and_preprocess_text("data/shakespeare.txt")
    processor = TextProcessor()
    processor.build_vocab(text)
    
    #Converting text to indices
    text_indices = processor.text_to_indices(text)
    print(f"{GREEN}Text converted to {len(text_indices)} training sequences")
    
    #Creating sequences
    sequences, target = create_sequence(text_indices, SEQUENCE_LENGTH)
    print(f"{GREEN}Created {len(sequences)} training sequences")
    
    #Creating the data loader
    dataset = torch.utils.data.TensorDataset(
        torch.tensor(sequences, dtype=torch.long),
        torch.tensor(target, dtype=torch.long)
    )
    
    data_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    #Initializing the model
    model = RNN(vocab_size=processor.vocab_size, embed_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(device)
    
    print(f"{PURPLE}Model parameters: {sum(p.numel() for p in model.parameters())}")
    
    #Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    #Training the model
    print(f"{PURPLE}Strated Training")
    losses = train_model(model, data_loader, criterion, optimizer, device, EPOCHS)
    
    #Generating sample text
    print(f"\n{BLUE}Generating sample text...")
    seed_texts = ["To be or not to be", "Romeo, Romeo", "All the world's a stage"]
    
    for seed in seed_texts:
        print(f"{CYAN}\nSeed: `{seed}`")
        generated = generate_model_output(model, processor, seed, length=300, temperature=0.8, device=device)
        print(f"{WHITE}generated")
        print("-" * 50)
    
    #Saving the model
    torch.save({
        "model_state_dict": model.state_dict(), 
        "processor": processor,
        "hyperparamters": {
            "vocab_size": processor.vocab_size,
            "batch_size": EMBED_SIZE,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS
        }
    }, "C:\Python_Programs\dl\shakespeare_rnn_model.pth")
    print(f"{GREEN}Model saved as `C:\Python_Programs\dl\shakespeare_rnn_model.pth`")

In [12]:
main()

Using Device: cpu
Text Length: 1115393 characters
Sample Text: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You...
Vocabulary Size: 65
Characters: 
   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k
Text converted to 1115393 training sequences
Created 1115343 training sequences
Model parameters: 946625
Strated Training


KeyboardInterrupt: 